# Retos · Problemas de entrevista

Problemas clásicos de entrevistas técnicas que mezclan lo visto en el resto de temas: diccionarios, conjuntos, sumas prefijas, *backtracking*, programación dinámica, grafos y montículos. Cada uno se resuelve primero con la idea ingenua, que suele ser la fuerza bruta, y después con la eficiente, indicando el coste de ambas y comprobando con `assert` que dan lo mismo.

In [1]:
import math
import random
import timeit

def mide(funcion, *args, repeticiones=3):
    return min(timeit.repeat(lambda: funcion(*args), repeat=repeticiones, number=1))

## 1. Two-sum

Dada una lista de enteros y un objetivo, devolver los índices de dos elementos distintos que sumen el objetivo, o `None` si no los hay. Ingenuo: probar todas las parejas, $O(n^2)$. Eficiente: recorrer la lista una vez guardando en un diccionario el índice de cada valor visto y buscar en él el complemento `objetivo - x`, $O(n)$ de tiempo y memoria.

In [2]:
def two_sum_ingenuo(numeros, objetivo):
    for i in range(len(numeros)):
        for j in range(i + 1, len(numeros)):
            if numeros[i] + numeros[j] == objetivo:
                return i, j
    return None

def two_sum(numeros, objetivo):
    vistos = {}
    for j, x in enumerate(numeros):
        if objetivo - x in vistos:
            return vistos[objetivo - x], j
        vistos[x] = j
    return None

print(two_sum([2, 7, 11, 15], 9), two_sum([3, 2, 4], 6), two_sum([3, 3], 6), two_sum([1, 2, 3], 7))

random.seed(1)
for _ in range(500):
    numeros = [random.randint(-20, 20) for _ in range(random.randint(0, 12))]
    objetivo = random.randint(-30, 30)
    a, b = two_sum_ingenuo(numeros, objetivo), two_sum(numeros, objetivo)
    assert (a is None) == (b is None)
    if b:
        i, j = b
        assert i != j and numeros[i] + numeros[j] == objetivo

pares = list(range(0, 4000, 2))  # un objetivo impar no tiene solución: peor caso
print(f"n = {len(pares)}: ingenuo {mide(two_sum_ingenuo, pares, 1) * 1000:.0f} ms, "
      f"diccionario {mide(two_sum, pares, 1) * 1000:.2f} ms")

(0, 1) (1, 2) (0, 1) None
n = 2000: ingenuo 63 ms, diccionario 0.13 ms


## 2. Anagramas agrupados

Agrupar las palabras de una lista que son anagramas entre sí. Ingenuo: comparar cada palabra con el representante de cada grupo ya formado, $O(n \cdot g \cdot k \log k)$ con $g$ grupos y palabras de longitud $k$, que es $O(n^2)$ cuando casi no hay anagramas. Eficiente: usar como clave de un diccionario la palabra con las letras ordenadas, $O(n \cdot k \log k)$.

In [3]:
from collections import defaultdict

def agrupa_ingenuo(palabras):
    grupos = []
    for palabra in palabras:
        for grupo in grupos:
            if sorted(grupo[0]) == sorted(palabra):
                grupo.append(palabra)
                break
        else:
            grupos.append([palabra])
    return grupos

def agrupa_anagramas(palabras):
    grupos = defaultdict(list)
    for palabra in palabras:
        grupos["".join(sorted(palabra))].append(palabra)
    return list(grupos.values())

def normaliza(grupos):
    return sorted(sorted(grupo) for grupo in grupos)

palabras = ["amor", "roma", "saco", "mora", "cosa", "pera", "ramo", "asco", "pare", "luz", "caso", "omar"]
for grupo in agrupa_anagramas(palabras):
    print(grupo)

random.seed(2)
for _ in range(300):
    lista = ["".join(random.choices("abc", k=random.randint(1, 4))) for _ in range(random.randint(0, 15))]
    assert normaliza(agrupa_ingenuo(lista)) == normaliza(agrupa_anagramas(lista))

['amor', 'roma', 'mora', 'ramo', 'omar']
['saco', 'cosa', 'asco', 'caso']
['pera', 'pare']
['luz']


## 3. Producto del array excepto sí mismo

Devolver una lista en la que la posición $i$ tenga el producto de todos los elementos salvo `numeros[i]`, sin dividir (dividir el producto total falla si hay ceros). Ingenuo: un producto por posición, $O(n^2)$. Eficiente: $r_i = \prod_{j<i} a_j \cdot \prod_{j>i} a_j$, el producto de los prefijos por el de los sufijos, que se calcula con dos pasadas en $O(n)$ y sin más memoria que la salida.

In [4]:
def producto_excepto_ingenuo(numeros):
    return [math.prod(numeros[:i] + numeros[i + 1:]) for i in range(len(numeros))]

def producto_excepto(numeros):
    resultado = [1] * len(numeros)
    izquierda = 1
    for i in range(len(numeros)):
        resultado[i] = izquierda
        izquierda *= numeros[i]
    derecha = 1
    for i in reversed(range(len(numeros))):
        resultado[i] *= derecha
        derecha *= numeros[i]
    return resultado

print(producto_excepto([1, 2, 3, 4]), producto_excepto([-1, 1, 0, -3, 3]))

random.seed(3)
for _ in range(300):
    numeros = [random.randint(-3, 3) for _ in range(random.randint(0, 9))]
    assert producto_excepto(numeros) == producto_excepto_ingenuo(numeros)

[24, 12, 8, 6] [0, 0, 9, 0, 0]


## 4. Secuencia consecutiva más larga

Dada una lista desordenada de enteros, encontrar la secuencia más larga de valores consecutivos que contiene (en `[100, 4, 200, 1, 3, 2]` es `1, 2, 3, 4`). Ingenuo: desde cada número, ir buscando `x + 1`, `x + 2`… con `in` sobre la lista, $O(n^3)$ en el peor caso. Ordenar ya lo deja en $O(n \log n)$. Eficiente: meter los números en un conjunto y empezar a contar solo desde los que no tienen anterior (`x - 1` no está), así cada número se visita como mucho dos veces: $O(n)$.

In [5]:
def consecutiva_ingenuo(numeros):
    mejor = 0
    for x in numeros:
        largo = 1
        while x + largo in numeros:
            largo += 1
        mejor = max(mejor, largo)
    return mejor

def consecutiva(numeros):
    conjunto = set(numeros)
    mejor, desde = 0, None
    for x in conjunto:
        if x - 1 not in conjunto:
            largo = 1
            while x + largo in conjunto:
                largo += 1
            if largo > mejor:
                mejor, desde = largo, x
    return mejor, list(range(desde, desde + mejor)) if mejor else []

print(consecutiva([100, 4, 200, 1, 3, 2]), consecutiva([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]), consecutiva([]))

random.seed(4)
for _ in range(500):
    numeros = [random.randint(-10, 10) for _ in range(random.randint(0, 12))]
    largo, secuencia = consecutiva(numeros)
    assert largo == consecutiva_ingenuo(numeros) and set(secuencia) <= set(numeros)

numeros = random.sample(range(10 ** 6), 3000)
print(f"n = {len(numeros)}: ingenuo {mide(consecutiva_ingenuo, numeros) * 1000:.0f} ms, "
      f"conjunto {mide(consecutiva, numeros) * 1000:.2f} ms")

(4, [1, 2, 3, 4]) (9, [0, 1, 2, 3, 4, 5, 6, 7, 8]) (0, [])
n = 3000: ingenuo 41 ms, conjunto 0.24 ms


## 5. Siguiente elemento mayor

Para cada elemento de una lista, encontrar el primero mayor que él a su derecha (o `None`); por ejemplo, para temperaturas diarias, el siguiente día más cálido. Ingenuo: recorrer hacia la derecha desde cada posición, $O(n^2)$. Eficiente: una pila con los índices que aún esperan respuesta, cuyos valores quedan en orden decreciente (pila monótona); al llegar un valor mayor que la cima, esa cima ya tiene respuesta. Cada índice entra y sale una vez: $O(n)$.

In [6]:
def siguiente_mayor_ingenuo(numeros):
    return [next((y for y in numeros[i + 1:] if y > x), None) for i, x in enumerate(numeros)]

def siguiente_mayor(numeros):
    resultado = [None] * len(numeros)
    pila = []
    for i, x in enumerate(numeros):
        while pila and numeros[pila[-1]] < x:
            resultado[pila.pop()] = x
        pila.append(i)
    return resultado

temperaturas = [18, 17, 21, 19, 16, 20, 24, 22]
print(siguiente_mayor(temperaturas))

random.seed(5)
for _ in range(500):
    numeros = [random.randint(0, 9) for _ in range(random.randint(0, 12))]
    assert siguiente_mayor(numeros) == siguiente_mayor_ingenuo(numeros)

decreciente = list(range(3000, 0, -1))  # peor caso del ingenuo: nadie tiene mayor a la derecha
print(f"n = {len(decreciente)}: ingenuo {mide(siguiente_mayor_ingenuo, decreciente) * 1000:.0f} ms, "
      f"pila {mide(siguiente_mayor, decreciente) * 1000:.2f} ms")

[21, 21, 24, 20, 20, 24, None, None]


n = 3000: ingenuo 54 ms, pila 0.16 ms


## 6. Intervalos solapados

Fusionar los intervalos cerrados `[inicio, fin]` que se solapan y calcular cuánto tiempo cubren en total (por ejemplo, las reuniones de una agenda). Ingenuo: buscar dos intervalos que se solapen, fusionarlos y volver a empezar hasta que no quede ninguna pareja, $O(n^3)$. Eficiente: ordenar por inicio y recorrerlos una vez, ampliando el último intervalo del resultado mientras el siguiente empiece antes de que acabe: $O(n \log n)$ por la ordenación.

In [7]:
def fusiona_ingenuo(intervalos):
    intervalos = [list(x) for x in intervalos]
    cambiado = True
    while cambiado:
        cambiado = False
        for i in range(len(intervalos)):
            for j in range(i + 1, len(intervalos)):
                (a, b), (c, d) = intervalos[i], intervalos[j]
                if a <= d and c <= b:
                    intervalos[i] = [min(a, c), max(b, d)]
                    del intervalos[j]
                    cambiado = True
                    break
            if cambiado:
                break
    return sorted(intervalos)

def fusiona(intervalos):
    resultado = []
    for inicio, fin in sorted(intervalos):
        if resultado and inicio <= resultado[-1][1]:
            resultado[-1][1] = max(resultado[-1][1], fin)
        else:
            resultado.append([inicio, fin])
    return resultado

reuniones = [[540, 600], [570, 630], [720, 780], [660, 690], [600, 615], [765, 800]]  # minutos desde las 0:00
agenda = fusiona(reuniones)
for inicio, fin in agenda:
    print(f"{inicio // 60:02}:{inicio % 60:02} - {fin // 60:02}:{fin % 60:02}")
print("minutos ocupados:", sum(fin - inicio for inicio, fin in agenda))

random.seed(6)
for _ in range(300):
    intervalos = []
    for _ in range(random.randint(0, 8)):
        a = random.randint(0, 20)
        intervalos.append([a, a + random.randint(0, 5)])
    assert fusiona(intervalos) == fusiona_ingenuo(intervalos)

09:00 - 10:30
11:00 - 11:30
12:00 - 13:20
minutos ocupados: 200


## 7. Rotar una matriz in situ

Girar 90° en sentido horario una matriz cuadrada sin crear otra. Con memoria extra $O(n^2)$ es inmediato: `zip(*matriz[::-1])`. In situ: trasponer intercambiando `m[i][j]` con `m[j][i]` y después invertir cada fila. Los dos son $O(n^2)$ de tiempo, pero el segundo solo usa memoria $O(1)$. Comprobar que coinciden y que cuatro giros dejan la matriz como estaba.

In [8]:
def rota_copia(matriz):
    return [list(fila) for fila in zip(*matriz[::-1])]

def rota(matriz):
    n = len(matriz)
    for i in range(n):
        for j in range(i + 1, n):
            matriz[i][j], matriz[j][i] = matriz[j][i], matriz[i][j]
    for fila in matriz:
        fila.reverse()

m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
primera_fila = m[0]
rota(m)
for fila in m:
    print(fila)
assert m[0] is primera_fila  # misma lista: no se ha creado otra matriz

random.seed(7)
for n in range(7):
    m = [[random.randint(0, 9) for _ in range(n)] for _ in range(n)]
    original = [fila[:] for fila in m]
    esperada = rota_copia(m)
    rota(m)
    assert m == esperada
    for _ in range(3):
        rota(m)
    assert m == original

[7, 4, 1]
[8, 5, 2]
[9, 6, 3]


## 8. Generar paréntesis válidos

Generar todas las cadenas con `n` pares de paréntesis bien formadas (en `12-clases-objetos` solo se comprobaban con una pila). Ingenuo: generar las $2^{2n}$ cadenas de `(` y `)` y quedarse con las equilibradas, $O(2^{2n} \cdot n)$. Eficiente: *backtracking* que solo añade `(` si quedan por abrir y `)` si hay alguno abierto, de modo que nunca construye una cadena inválida: $O(C_n \cdot n)$, con $C_n = \frac{1}{n+1}\binom{2n}{n}$ el número de Catalan.

In [9]:
from itertools import product

def equilibrada(cadena):
    abiertos = 0
    for c in cadena:
        abiertos += 1 if c == "(" else -1
        if abiertos < 0:
            return False
    return abiertos == 0

def parentesis_ingenuo(n):
    return ["".join(c) for c in product("()", repeat=2 * n) if equilibrada(c)]

def parentesis(n):
    resultado = []

    def coloca(cadena, abiertos, cerrados):
        if len(cadena) == 2 * n:
            resultado.append(cadena)
            return
        if abiertos < n:
            coloca(cadena + "(", abiertos + 1, cerrados)
        if cerrados < abiertos:
            coloca(cadena + ")", abiertos, cerrados + 1)

    coloca("", 0, 0)
    return resultado

print(parentesis(3))
print([len(parentesis(n)) for n in range(1, 12)])

for n in range(9):
    assert parentesis(n) == parentesis_ingenuo(n)
for n in range(12):
    assert len(parentesis(n)) == math.comb(2 * n, n) // (n + 1)

print(f"n = 8: ingenuo {mide(parentesis_ingenuo, 8) * 1000:.0f} ms, backtracking {mide(parentesis, 8) * 1000:.1f} ms")

['((()))', '(()())', '(())()', '()(())', '()()()']
[1, 2, 5, 14, 42, 132, 429, 1430, 4862, 16796, 58786]


n = 8: ingenuo 15 ms, backtracking 0.6 ms


## 9. Decodificar mensajes

Un mensaje se codifica con A → 1, B → 2, …, Z → 26. Contar de cuántas formas se puede decodificar una cadena de cifras (`"226"` es BZ, VF o BBF). Ingenuo: recursión que prueba a tomar una o dos cifras, $O(2^n)$. Eficiente: programación dinámica

$$f(i) = [s_i \neq 0]\, f(i-1) + [10 \le s_{i-1} s_i \le 26]\, f(i-2),$$

que solo necesita los dos valores anteriores: $O(n)$ de tiempo y $O(1)$ de memoria. Listar también las decodificaciones de un mensaje corto.

In [10]:
def decodificaciones_ingenuo(cifras):
    if not cifras:
        return 1
    if cifras[0] == "0":
        return 0
    total = decodificaciones_ingenuo(cifras[1:])
    if len(cifras) >= 2 and int(cifras[:2]) <= 26:
        total += decodificaciones_ingenuo(cifras[2:])
    return total

def decodificaciones(cifras):
    previo, actual = 0, 1
    for i, c in enumerate(cifras):
        nuevo = actual if c != "0" else 0
        if i > 0 and 10 <= int(cifras[i - 1:i + 1]) <= 26:
            nuevo += previo
        previo, actual = actual, nuevo
    return actual

def lista_decodificaciones(cifras):
    if not cifras:
        return [""]
    resultado = []
    for largo in (1, 2):
        trozo = cifras[:largo]
        if len(trozo) == largo and trozo[0] != "0" and int(trozo) <= 26:
            letra = chr(ord("A") + int(trozo) - 1)
            resultado += [letra + resto for resto in lista_decodificaciones(cifras[largo:])]
    return resultado

for cifras in ["12", "226", "06", "10", "11106", "2611055971756562"]:
    print(f"{cifras:>17}: {decodificaciones(cifras):>3}", lista_decodificaciones(cifras)[:5])

random.seed(9)
for _ in range(500):
    cifras = "".join(random.choices("0123456789", weights=[1, 4, 4, 1, 1, 1, 1, 1, 1, 1], k=random.randint(0, 16)))
    assert decodificaciones(cifras) == decodificaciones_ingenuo(cifras) == len(lista_decodificaciones(cifras))

fib = [0, 1]
while len(fib) < 70:
    fib.append(fib[-1] + fib[-2])
assert decodificaciones("1" * 60) == fib[61]  # sin la DP serían 2.5e12 llamadas
print(decodificaciones("1" * 60))

               12:   2 ['AB', 'L']
              226:   3 ['BBF', 'BZ', 'VF']
               06:   0 []
               10:   1 ['J']
            11106:   2 ['AAJF', 'KJF']
 2611055971756562:   4 ['BFAJEEIGAGEFEFB', 'BFAJEEIGQEFEFB', 'ZAJEEIGAGEFEFB', 'ZAJEEIGQEFEFB']
2504730781961


## 10. Camino mínimo en un triángulo

Bajar desde la cima de un triángulo de números hasta la base pasando en cada fila a uno de los dos números adyacentes de la siguiente, minimizando la suma. Ingenuo: probar los $2^{n-1}$ caminos. Eficiente: programación dinámica de abajo arriba,

$$m_{i,j} = t_{i,j} + \min(m_{i+1,j},\ m_{i+1,j+1}),$$

que con una sola fila es $O(n^2)$ de tiempo y $O(n)$ de memoria. Reconstruir también el camino guardando la tabla entera.

In [11]:
def triangulo_ingenuo(t):
    mejor = math.inf
    for pasos in product((0, 1), repeat=len(t) - 1):
        j, suma = 0, t[0][0]
        for fila, paso in zip(t[1:], pasos):
            j += paso
            suma += fila[j]
        mejor = min(mejor, suma)
    return mejor

def triangulo(t):
    minimos = t[-1][:]
    for fila in reversed(t[:-1]):
        minimos = [x + min(minimos[j], minimos[j + 1]) for j, x in enumerate(fila)]
    return minimos[0]

def camino_minimo(t):
    m = [fila[:] for fila in t]
    for i in range(len(t) - 2, -1, -1):
        for j in range(i + 1):
            m[i][j] += min(m[i + 1][j], m[i + 1][j + 1])
    camino, j = [], 0
    for i, fila in enumerate(t):
        if i > 0 and m[i][j + 1] < m[i][j]:
            j += 1
        camino.append(fila[j])
    return m[0][0], camino

t = [[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]
for fila in t:
    print(" ".join(map(str, fila)).center(9))
print(camino_minimo(t))

random.seed(10)
for _ in range(200):
    t = [[random.randint(-9, 9) for _ in range(i + 1)] for i in range(random.randint(1, 12))]
    total, camino = camino_minimo(t)
    assert triangulo(t) == triangulo_ingenuo(t) == total == sum(camino)

    2    
   3 4   
  6 5 7  
 4 1 8 3 
(11, [2, 3, 5, 1])


## 11. Número de islas

En una rejilla de `1` (tierra) y `0` (agua), contar las islas: grupos de tierra unidos en horizontal o vertical (en `17-grafos` se cuentan con un *flood fill* recursivo). Ingenuo: para cada casilla de tierra calcular su isla entera con una BFS y guardar las islas distintas en un conjunto, $O((RC)^2)$. Eficiente: recorrer la rejilla una vez y, al encontrar tierra sin visitar, sumar una isla y marcarla entera con una DFS iterativa (con una pila, sin límite de recursión), $O(RC)$. Comprobar contra `scipy.ndimage.label`.

In [12]:
from collections import deque

import numpy as np
from scipy import ndimage

VECINOS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def isla_de(rejilla, f, c):
    isla, cola = {(f, c)}, deque([(f, c)])
    while cola:
        y, x = cola.popleft()
        for dy, dx in VECINOS:
            v = (y + dy, x + dx)
            if 0 <= v[0] < len(rejilla) and 0 <= v[1] < len(rejilla[0]) and rejilla[v[0]][v[1]] == 1 and v not in isla:
                isla.add(v)
                cola.append(v)
    return frozenset(isla)

def islas_ingenuo(rejilla):
    return len({isla_de(rejilla, f, c) for f, fila in enumerate(rejilla) for c, x in enumerate(fila) if x == 1})

def islas(rejilla):
    filas, columnas = len(rejilla), len(rejilla[0])
    vistas, total = set(), 0
    for f in range(filas):
        for c in range(columnas):
            if rejilla[f][c] == 1 and (f, c) not in vistas:
                total += 1
                vistas.add((f, c))
                pila = [(f, c)]
                while pila:
                    y, x = pila.pop()
                    for dy, dx in VECINOS:
                        v = (y + dy, x + dx)
                        if 0 <= v[0] < filas and 0 <= v[1] < columnas and rejilla[v[0]][v[1]] == 1 and v not in vistas:
                            vistas.add(v)
                            pila.append(v)
    return total

mapa = ["11000110", "11000100", "00100001", "00011011", "10000011"]
rejilla = [[int(x) for x in fila] for fila in mapa]
print("\n".join(fila.replace("1", "█").replace("0", "·") for fila in mapa))
print("islas:", islas(rejilla))

random.seed(11)
for _ in range(100):
    rejilla = [[int(random.random() < 0.45) for _ in range(8)] for _ in range(8)]
    assert islas(rejilla) == islas_ingenuo(rejilla) == ndimage.label(np.array(rejilla))[1]

grande = [[int(random.random() < 0.55) for _ in range(300)] for _ in range(300)]
assert islas(grande) == ndimage.label(np.array(grande))[1]
print("300 x 300:", islas(grande), "islas")

██···██·
██···█··
··█····█
···██·██
█·····██
islas: 6
300 x 300: 3944 islas


## 12. Palabras en una sopa de letras

Buscar en una sopa de letras todas las apariciones de una lista de palabras en las 8 direcciones. Ingenuo: para cada palabra, casilla y dirección, comparar letra a letra, $O(P \cdot RC \cdot 8L)$ con $P$ palabras de longitud hasta $L$. Eficiente: meter las palabras en un *trie* (diccionarios anidados) y, desde cada casilla y dirección, avanzar por él mientras lo leído sea prefijo de alguna palabra: $O(RC \cdot 8L)$, sin depender del número de palabras.

In [13]:
DIRECCIONES = {(0, 1): "→", (0, -1): "←", (1, 0): "↓", (-1, 0): "↑",
               (1, 1): "↘", (1, -1): "↙", (-1, 1): "↗", (-1, -1): "↖"}

def busca_ingenuo(sopa, palabras):
    filas, columnas = len(sopa), len(sopa[0])
    encontradas = set()
    for palabra in palabras:
        for f in range(filas):
            for c in range(columnas):
                for df, dc in DIRECCIONES:
                    if all(0 <= f + k * df < filas and 0 <= c + k * dc < columnas and sopa[f + k * df][c + k * dc] == letra
                           for k, letra in enumerate(palabra)):
                        encontradas.add((palabra, f, c, df, dc))
    return encontradas

def crea_trie(palabras):
    raiz = {}
    for palabra in palabras:
        nodo = raiz
        for letra in palabra:
            nodo = nodo.setdefault(letra, {})
        nodo["$"] = palabra  # fin de palabra: '$' no aparece en la sopa
    return raiz

def busca(sopa, palabras):
    trie = crea_trie(palabras)
    filas, columnas = len(sopa), len(sopa[0])
    encontradas = set()
    for f in range(filas):
        for c in range(columnas):
            for df, dc in DIRECCIONES:
                nodo, y, x = trie, f, c
                while 0 <= y < filas and 0 <= x < columnas and sopa[y][x] in nodo:
                    nodo = nodo[sopa[y][x]]
                    if "$" in nodo:
                        encontradas.add((nodo["$"], f, c, df, dc))
                    y, x = y + df, x + dc
    return encontradas

SOPA = [
    "PYTHONXA",
    "LQBUCLEL",
    "IWRGRAFO",
    "SDICTZEP",
    "TCLASEMI",
    "AKJVRTUL",
    "OCIFARGA",
    "NOHTYPSX",
]
PALABRAS = ["PYTHON", "BUCLE", "GRAFO", "GRAFICO", "LISTA", "PILA", "CLASE", "DICT", "COLA"]

encontradas = busca(SOPA, PALABRAS)
for palabra, f, c, df, dc in sorted(encontradas):
    print(f"{palabra:<8} fila {f}, columna {c} {DIRECCIONES[df, dc]}")
print("no están:", sorted(set(PALABRAS) - {e[0] for e in encontradas}))
assert encontradas == busca_ingenuo(SOPA, PALABRAS)

random.seed(12)
for _ in range(200):
    sopa = ["".join(random.choices("AB", k=5)) for _ in range(5)]
    palabras = list({"".join(random.choices("AB", k=random.randint(1, 4))) for _ in range(4)})
    assert busca(sopa, palabras) == busca_ingenuo(sopa, palabras)

BUCLE    fila 1, columna 2 →
CLASE    fila 4, columna 1 →
DICT     fila 3, columna 1 →
GRAFICO  fila 6, columna 6 ←
GRAFO    fila 2, columna 3 →
LISTA    fila 1, columna 0 ↓
PILA     fila 3, columna 7 ↓
PYTHON   fila 0, columna 0 →
PYTHON   fila 7, columna 5 ←
no están: ['COLA']


## 13. Caché LRU con `OrderedDict`

Caché de capacidad fija que, al llenarse, expulsa la clave usada hace más tiempo (en `12-clases-objetos` se construye a mano con una lista doblemente enlazada). Ingenuo: un diccionario y una lista con el orden de uso, donde llevar una clave al final con `list.remove` cuesta $O(n)$. Eficiente: `OrderedDict` mantiene el orden de inserción y tiene `move_to_end` y `popitem(last=False)` en $O(1)$. Comparar las dos con una secuencia aleatoria de operaciones y contar los fallos de caché contra `functools.lru_cache`.

In [14]:
from collections import OrderedDict
from functools import lru_cache

class LRUIngenua:
    def __init__(self, capacidad):
        self.capacidad, self.datos, self.orden = capacidad, {}, []

    def get(self, clave):
        if clave not in self.datos:
            return None
        self.orden.remove(clave)
        self.orden.append(clave)
        return self.datos[clave]

    def put(self, clave, valor):
        if clave in self.datos:
            self.orden.remove(clave)
        elif len(self.datos) == self.capacidad:
            del self.datos[self.orden.pop(0)]
        self.datos[clave] = valor
        self.orden.append(clave)

class LRU:
    def __init__(self, capacidad):
        self.capacidad, self.datos = capacidad, OrderedDict()

    def get(self, clave):
        if clave not in self.datos:
            return None
        self.datos.move_to_end(clave)
        return self.datos[clave]

    def put(self, clave, valor):
        if clave in self.datos:
            self.datos.move_to_end(clave)
        self.datos[clave] = valor
        if len(self.datos) > self.capacidad:
            self.datos.popitem(last=False)

cache = LRU(2)
cache.put("a", 1)
cache.put("b", 2)
cache.get("a")
cache.put("c", 3)
print(list(cache.datos.items()), cache.get("b"))  # 'b' era la menos usada

random.seed(13)
for capacidad in range(1, 6):
    ingenua, eficiente = LRUIngenua(capacidad), LRU(capacidad)
    for _ in range(2000):
        clave = random.randint(0, 7)
        if random.random() < 0.5:
            assert ingenua.get(clave) == eficiente.get(clave)
        else:
            ingenua.put(clave, clave * 10)
            eficiente.put(clave, clave * 10)
        assert ingenua.orden == list(eficiente.datos)

def fallos(capacidad, claves):
    cache, total = LRU(capacidad), 0
    for clave in claves:
        if cache.get(clave) is None:
            total += 1
            cache.put(clave, clave * clave)
    return total

claves = [random.randint(0, 9) for _ in range(1000)]

@lru_cache(maxsize=4)
def cuadrado(k):
    return k * k

for clave in claves:
    cuadrado(clave)
assert fallos(4, claves) == cuadrado.cache_info().misses
print(cuadrado.cache_info())

[('a', 1), ('c', 3)] None
CacheInfo(hits=413, misses=587, maxsize=4, currsize=4)


## 14. Mediana de un flujo

Recibir números uno a uno y dar la mediana tras cada uno. Ingenuo: mantener la lista ordenada con `bisect.insort`, $O(n)$ por inserción por el desplazamiento de elementos. Eficiente: dos montículos, uno de máximos con la mitad pequeña (con `heapq` y los valores negados) y otro de mínimos con la mitad grande, equilibrados para que difieran como mucho en uno: insertar es $O(\log n)$ y la mediana está en las cimas, $O(1)$. Comprobar con `statistics.median`.

In [15]:
import bisect
import heapq
import statistics

class MedianaIngenua:
    def __init__(self):
        self.valores = []

    def agrega(self, x):
        bisect.insort(self.valores, x)

    def mediana(self):
        n = len(self.valores)
        return self.valores[n // 2] if n % 2 else (self.valores[n // 2 - 1] + self.valores[n // 2]) / 2

class Mediana:
    def __init__(self):
        self.bajos, self.altos = [], []  # bajos guarda los valores negados: montículo de máximos

    def agrega(self, x):
        if self.bajos and x > -self.bajos[0]:
            heapq.heappush(self.altos, x)
        else:
            heapq.heappush(self.bajos, -x)
        if len(self.bajos) > len(self.altos) + 1:
            heapq.heappush(self.altos, -heapq.heappop(self.bajos))
        elif len(self.altos) > len(self.bajos):
            heapq.heappush(self.bajos, -heapq.heappop(self.altos))

    def mediana(self):
        if len(self.bajos) > len(self.altos):
            return -self.bajos[0]
        return (-self.bajos[0] + self.altos[0]) / 2

flujo = Mediana()
for x in [5, 15, 1, 3, 8, 7, 9, 10]:
    flujo.agrega(x)
    print(f"llega {x:>2}: mediana {flujo.mediana()}")

random.seed(14)
ingenua, eficiente, vistos = MedianaIngenua(), Mediana(), []
for _ in range(500):
    x = random.randint(-100, 100)
    vistos.append(x)
    ingenua.agrega(x)
    eficiente.agrega(x)
    assert eficiente.mediana() == ingenua.mediana() == statistics.median(vistos)
    assert abs(len(eficiente.bajos) - len(eficiente.altos)) <= 1

llega  5: mediana 5
llega 15: mediana 10.0
llega  1: mediana 5
llega  3: mediana 4.0
llega  8: mediana 5
llega  7: mediana 6.0
llega  9: mediana 7
llega 10: mediana 7.5
